In [15]:
import warnings
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

### LOAD DATA

In [16]:
# Loading the data
df = pd.read_csv("traffic_data_gen.csv")

# Clean target labels
df["traffic_level"] = df["traffic_level"].astype(str).str.strip().str.lower()

# Map to Binary: 'low' -> 'normal' (0), 'medium'/'high' -> 'congested' (1)
target_mapping = {"low": 0, "medium": 1, "high": 1}
df["traffic_binary"] = df["traffic_level"].map(target_mapping)

# Drop any unmapped or missing target rows
df = df.dropna(subset=["traffic_binary"]).copy()
df["traffic_binary"] = df["traffic_binary"].astype(int)

# Feature extraction: Extract hour if observation_time exists
if "observation_time" in df.columns:
    df["obs_hour"] = pd.to_datetime(df["observation_time"], errors="coerce").dt.hour

### DEFINE TARGET & FEATURES

In [17]:
# Explicitly drop target columns, leaky columns, and raw text/datetime columns
drop_cols = [
    'traffic_level',
    'traffic_binary',
    'traffic_pattern',
    'traffic_score',
    'congestion_location',
    'traffic_level_encoded',
    'date',
    'observation_time',
    'time_block',
    'day_encoded',
    'traffic_pattern_encoded',
    'congestion_location_encoded'
]
X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore").copy()
y = df["traffic_binary"]

numeric_features = X.select_dtypes(include=["int64", "float64", "Int64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Target Distribution:\n{y.value_counts(normalize=True).round(3)}")

Target Distribution:
traffic_binary
1    0.667
0    0.333
Name: proportion, dtype: float64


In [18]:
"""X = X.copy()
X['obs_hour'] = df['obs_hour'].astype('int64')

numeric_features = X.select_dtypes(include=["int64", "float64", "Int64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()"""

X['obs_hour'] = X['obs_hour'].astype('category')

# THEN compute the feature lists
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X.select_dtypes(include=["int64", "float64", "Int64"]).columns.tolist()

# rebuild lr_pipe's ColumnTransformer with the updated lists, retrain

print(f"Numeric Features: {numeric_features}")  # should now include 'obs_hour'
print(f"Categorical Features: {categorical_features}")

Numeric Features: ['temperature', 'rain_chance']
Categorical Features: ['road', 'fixed_corridor', 'day', 'weather', 'obs_hour']


### IDENTIFY COLUMN TYPE

In [19]:
numeric_features = X.select_dtypes(include=['int64', 'float64', 'Int64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"\nTarget distribution:\n{y.value_counts()}")

Numeric features (2): ['temperature', 'rain_chance']
Categorical features (5): ['road', 'fixed_corridor', 'day', 'weather', 'obs_hour']

Target distribution:
traffic_binary
1    1000
0     500
Name: count, dtype: int64


### PREPROCESSING PIPELINE

In [20]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            categorical_features,
        ),
    ]
)

### TRAIN / TEST SPLIT (Stratified!)

In [21]:
# Stratify ensures low/medium/high appear in both sets.
# Encode y for compatibility across both models (especially XGBoost)
"""label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)"""

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 5-fold CV is now possible with consolidated minority instances
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Compute positive class scale weight for XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

### MODEL 1: LOGISTIC REGRESSION (Baseline)

In [22]:
lr_pipe = Pipeline(
    [
        ("prep", preprocessor),
        (
            "clf",
            LogisticRegression(
                solver="lbfgs",
                max_iter=1000,
                C=0.5,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

In [23]:
# Cross-validation (more reliable than single split on small data)
lr_cv = cross_val_score(lr_pipe, X_train, y_train, cv=cv, scoring="f1_macro")
print(f"\n--- Logistic Regression ---")
print(f"CV F1 (macro): {lr_cv.mean():.3f} (+/- {lr_cv.std():.3f})")

lr_pipe.fit(X_train, y_train)
lr_pred = lr_pipe.predict(X_test)
print(
    classification_report(
        y_test, lr_pred, target_names=["Normal (0)", "Congested (1)"], zero_division=0
    )
)
print("Balanced Accuracy:", balanced_accuracy_score(y_test, lr_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, lr_pred))


--- Logistic Regression ---
CV F1 (macro): 0.843 (+/- 0.013)
               precision    recall  f1-score   support

   Normal (0)       0.70      0.99      0.82       150
Congested (1)       1.00      0.79      0.88       300

     accuracy                           0.86       450
    macro avg       0.85      0.89      0.85       450
 weighted avg       0.90      0.86      0.86       450

Balanced Accuracy: 0.8899999999999999
Confusion Matrix:
 [[149   1]
 [ 64 236]]


### Save model artifacts

In [24]:
# Save the fitted end-to-end pipeline and the label encoder
joblib.dump(lr_pipe, "traffic_binary_lr_pipeline.pkl")
print("\nBinary pipelines saved as .pkl files successfully.")


Binary pipelines saved as .pkl files successfully.


In [25]:
check = df.loc[y_test.index].copy()
check['pred'] = lr_pipe.predict(X_test)  # or however you get preds
print(check.groupby('obs_hour').apply(
    lambda g: pd.Series({'n': len(g), 'err_rate': (g['pred'] != y_test.loc[g.index]).mean()})
))

             n  err_rate
obs_hour                
6          7.0  0.000000
7         16.0  0.000000
8         39.0  0.000000
9         32.0  0.000000
10        18.0  0.000000
11        23.0  0.000000
12        31.0  0.516129
13        23.0  0.434783
14        39.0  0.461538
15        14.0  0.000000
16        42.0  0.000000
17        32.0  0.000000
18        25.0  0.000000
19        33.0  0.000000
20        43.0  0.488372
21        14.0  0.000000
22        19.0  0.000000
